In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

In [ ]:
    
for num_nodes in  [48,32,16,8,4]:
    
    # Fetch all existing instance names matching tsm-sc-*
    fetch_cmd = f'''
    gcloud compute instances list \
        --filter="name~'tsm-sc-'" \
        --format="value(name)"
    '''
    instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
    instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries
    
    print("\n➡ Existing instances to delete:", instances_to_delete)
    
    if instances_to_delete:
        # Use parallel deletion
        def delete_instance(instance_name):
            cmd = f'''
            gcloud compute instances delete {instance_name} \
                --zone={zone} \
                --project={project} \
                --quiet
            '''
            print(f"Deleting: {instance_name}")
            return subprocess.call(cmd, shell=True)
    
        with concurrent.futures.ThreadPoolExecutor(max_workers=48) as executor:
            futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
            concurrent.futures.wait(futures)
    
        print("🧹 All old tsm-sc-* instances deleted.\n")
    else:
        print("✔ No previous instances found to delete.\n")
    
    

    
    project = "ucr-ursa-major-lesani-lab"
    zone = "us-central1-c"
    machine_type = "e2-highmem-2"
    image_family = "tsm-sc-family"  # your custom image
    subnet = "default"
    gcp_username = "tejas"
    

    # Cleanup any existing instances with same prefix
    os.system(f'gcloud compute instances delete --zone={zone} --quiet '
              f'$(gcloud compute instances list --filter="name~\'tsm-sc-\'" --format="value(name)")')
    
    # Create commands list
    commands = []
    
    for i in range(num_nodes):
        cmd = f'''
        gcloud compute instances create tsm-sc-{i:03} \
            --project={project} \
            --zone={zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=961693926925-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
    
    
    def run_command(command):
        print(f"Running: {command}")
        return subprocess.call(command, shell=True)
    
    
    # #Parallel instance creation
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=48) as executor:
        futures = [executor.submit(run_command, cmd) for cmd in commands]
        concurrent.futures.wait(futures)
    
    print("All instances launched.")
    

    # Wait a bit for IPs to propagate
    import time
    time.sleep(30)
    

    # Get IPs
    os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
              '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')
    
    with open('tsm_ips.txt', 'r') as f:
        iplist = [line.strip() for line in f.readlines()]
    
    print("🎯 Instance IPs:", iplist)
    

    os.system('git add .; git commit -m "testing"; git push')
   

    n_collection = 100
    os.system('make -j8')
    
    

    def kill_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill -9 stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    
    
    
    def git_pull_stellar(i):
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    git pull"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=48)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    import shutil
    
    if os.path.exists('../stellar-private'):
        
        shutil.rmtree('../stellar-private')
    os.mkdir('../stellar-private')
    
    
    os.system('cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh')
    
    os.system('cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; ./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh')
    
    # --- Configuration ---
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg" 
    
    # --- The os.system() Command ---
    # This command prepends the line to the target_file on your local machine.
    os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')
    
    print(f"The line '{line_to_add}' has been prepended to {target_file}.")
    

    def compile_stellar(i):
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    make -j16; cd; sudo rm -r stellar-private"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=48)(delayed(compile_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    def clean_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas; \
    sudo rm -r stellar-private; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(clean_stellar_private)(i) for i in range(num_nodes))
    

    def copy_folder_to_instance(i, source_folder = '/home/tejas/stellar-private',destination_path = '/home/tejas/stellar-private' ):
        """
        Constructs and executes the gcloud compute scp command to copy a folder
        to a specific GCP instance.
        """
        instance_name = f"tsm-sc-{i:03}"
        
        # The --recurse flag is crucial for copying folders
        # The format is: gcloud compute scp --recurse [LOCAL_SRC] [USER]@[INSTANCE_NAME]:[REMOTE_DEST]
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{source_folder}" "{instance_name}:{destination_path}"'
    
        print(f"Executing command for {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Command for {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    
    results = Parallel(n_jobs=48)(
        delayed(copy_folder_to_instance)(i) for i in range(num_nodes)
    )
    
    

    
    # def setup_stellar_private(i):
    #     command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    # cd /home/tejas; \
    # cd stellar-private; chmod +x gcp_setup_stellar_private.sh; \
    # ./gcp_setup_stellar_private.sh;"'
    #     print(command)
    #     output = os.system(command)
    #     print(output)
    
    # # Execute in parallel like your example
    # results = Parallel(n_jobs=20)(delayed(setup_stellar_private)(i) for i in range(len(iplist)))
    # print(results)
    
    def run_stellar_private(i):
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        instance_name = f"tsm-sc-{i:03}"
        
        # ----------------------------------------------------------------------------------
        # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
        # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
        # '< /dev/null' ensures the process doesn't wait for input.
        # ----------------------------------------------------------------------------------
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
    > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
        
        # os.system should now return immediately because the remote shell exits
        output = os.system(command)
        print(f"Return code for {instance_name}: {output}")


        
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))

    # Corrected Loop (to run 0, 1, 2, 3)
    results = Parallel(n_jobs=48)(delayed(run_stellar_private)(i) for i in range(num_nodes))
    # results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in [3,2,1,0])
    

    # time.sleep(3)
    # for i in range(num_nodes):
    
        # run_stellar_private(num_nodes-i-1)
        # time.sleep(2)
    # run_stellar_private(0)
    print(results)
    print("All SSH commands executed. Nodes should be starting up in the background.")
    
    time.sleep(200)
    

    
    
    def kill_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    remote_base_folder = "/home/tejas/stellar-private" # The base path on the GCP instance
    local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_"+str(n_collection)+"_rounds_" + str(num_nodes) + "" 
    # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_" + str(num_nodes) + ""
    
    # Ensure the local base destination directory exists
    os.makedirs(local_base_destination, exist_ok=True)
    
    
    def copy_folder_from_instance(i):
        """
        Constructs and executes the gcloud compute scp command to copy a specific 
        nodeN folder from instance i to a local folder named after the instance.
        """
        instance_name = f"tsm-sc-{i:03}"
        
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        node_folder = f"node{node_number}"
    
        # 1. Define the specific REMOTE source path on the instance
        # Example: /home/tejas/stellar-private/node1
        remote_source_path = os.path.join(remote_base_folder, node_folder)
        
        # 2. Define the LOCAL destination path
        # We'll use the instance name for the subfolder to keep backups separate
        local_destination_path = os.path.join(local_base_destination, instance_name)
        os.makedirs(local_destination_path, exist_ok=True)
        
        # The SCp command requires the remote path to be formatted as:
        # [INSTANCE_NAME]:[REMOTE_SRC]
        remote_source = f"{instance_name}:{remote_source_path}"
        
        # The command reverses the source (remote) and destination (local)
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{remote_source}" "{local_destination_path}"'
    
        print(f"Executing command to copy {node_folder} from {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Copy from {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    # ---
    # Execute the copy operation in parallel
    # ---
    
    results = Parallel(n_jobs=48)(
        delayed(copy_folder_from_instance)(i) for i in range(3)
    )
    
    print("\n--- Summary of Download Results ---")
    print(results)
    





➡ Existing instances to delete: ['tsm-sc-000', 'tsm-sc-001', 'tsm-sc-002', 'tsm-sc-003', 'tsm-sc-004', 'tsm-sc-005', 'tsm-sc-006', 'tsm-sc-007', 'tsm-sc-008', 'tsm-sc-009', 'tsm-sc-010', 'tsm-sc-011', 'tsm-sc-012', 'tsm-sc-013', 'tsm-sc-014', 'tsm-sc-015', 'tsm-sc-016', 'tsm-sc-017', 'tsm-sc-018', 'tsm-sc-019', 'tsm-sc-020', 'tsm-sc-021', 'tsm-sc-022', 'tsm-sc-023', 'tsm-sc-024', 'tsm-sc-025', 'tsm-sc-026', 'tsm-sc-027', 'tsm-sc-028', 'tsm-sc-029', 'tsm-sc-030', 'tsm-sc-031', 'tsm-sc-032', 'tsm-sc-033', 'tsm-sc-034', 'tsm-sc-035', 'tsm-sc-036', 'tsm-sc-037', 'tsm-sc-038', 'tsm-sc-039', 'tsm-sc-040', 'tsm-sc-041', 'tsm-sc-042', 'tsm-sc-043', 'tsm-sc-044', 'tsm-sc-045', 'tsm-sc-046', 'tsm-sc-047']
🧹 All old tsm-sc-* instances deleted.



Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-004].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-008].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-009].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-010].


Running: gcloud compute instances create tsm-sc-000             --project=ucr-ursa-major-lesani-lab             --zone=us-central1-c             --machine-type=e2-highmem-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=961693926925-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-shielded-secure-boot             --shielded-vtpm    

Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-004].
Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-004  us-central1-c  e2-highmem-2               10.128.0.74  136.111.199.43  RUNNING
NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-001  us-central1-c  e2-highmem-2               10.128.0.65  35.238.189.239  RUNNING


...

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-016" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-016: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-002" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-008" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-019" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-019: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-030" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-014" --project "ucr-ursa-major-lesani-lab" --command

Exception ignored in: <function ResourceTracker.__del__ at 0x73afcf18e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7d8ff518e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-000: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-015" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-022" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-022: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-026" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-023" --project "ucr-ursa-major-lesani-lab" --command

Exception ignored in: <function ResourceTracker.__del__ at 0x7c5c99386020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x793043d82020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
All SSH commands executed. Nodes should be starting up in the background.
